# Challenge 2: Make the Quantum Coin Biased
Qiskit Fall Fest 2026 – IBM Quantum, Vishnu Institute of Technology

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram

try:
    from qiskit_aer import AerSimulator
    backend = AerSimulator()
except ImportError:
    from qiskit.providers.basic_provider import BasicSimulator
    backend = BasicSimulator()

SHOTS = 100

# Gate colours: [fill, text]
circuit_style = {
    "displaycolor": {
        "ry":      ["#003399", "#FFFFFF"],
        "h":       ["#003399", "#FFFFFF"],
        "measure": ["#A9A9A9", "#000000"],
    }
}

def run(qc):
    """Run a circuit for 100 shots and return the counts."""
    return backend.run(transpile(qc, backend), shots=SHOTS).result().get_counts()

## Prediction (before running)
I apply a single **RY(π/3)** gate to |0⟩. For RY(θ), P(0) = cos²(θ/2) and P(1) = sin²(θ/2).

- P(0) = cos²(π/6) = **75%**, P(1) = sin²(π/6) = **25%**
- So **0 should occur more often**, roughly 75 vs 25 out of 100 shots (with some random variation).
- Changing the gate angle changes the bias: a larger angle moves probability toward 1, a smaller one keeps it near 0.

## Main circuit: bias towards 0 (RY(π/3))

In [ ]:
qc = QuantumCircuit(1, 1)       # qubit starts in |0>
qc.ry(np.pi / 3, 0)
qc.measure(0, 0)

qc.draw(output="mpl", style=circuit_style)

## Run for 100 shots

In [ ]:
counts = run(qc)
print("Counts:", counts)
print(f"P(0) = {counts.get('0', 0)/SHOTS:.2f}   P(1) = {counts.get('1', 0)/SHOTS:.2f}")
print("Predicted: P(0) = 0.75   P(1) = 0.25")

## Histogram

In [ ]:
plot_histogram(counts, title="RY(π/3): biased towards 0",
               color="#003399", figsize=(7, 5))

## Bonus: a second circuit where 1 occurs more often
**Prediction:** RY(2π/3) gives P(0) = cos²(π/3) = 25% and P(1) = sin²(π/3) = **75%**, so **1** should now dominate.

In [ ]:
qc_bonus = QuantumCircuit(1, 1)
qc_bonus.ry(2 * np.pi / 3, 0)
qc_bonus.measure(0, 0)

qc_bonus.draw(output="mpl", style=circuit_style)

In [ ]:
counts_bonus = run(qc_bonus)
print("Counts:", counts_bonus)
print(f"P(0) = {counts_bonus.get('0', 0)/SHOTS:.2f}   P(1) = {counts_bonus.get('1', 0)/SHOTS:.2f}")

In [ ]:
plot_histogram(counts_bonus, title="RY(2π/3): biased towards 1",
               color="#CC3300", figsize=(7, 5))

## Compare the two biased circuits

In [ ]:
plot_histogram([counts, counts_bonus],
               legend=["RY(π/3): favours 0", "RY(2π/3): favours 1"],
               color=["#003399", "#CC3300"],
               title="Bias towards 0 vs bias towards 1", figsize=(8, 5))

## Different degrees of bias (sweeping the RY angle)

In [ ]:
angles = {"π/6": np.pi/6, "π/3": np.pi/3, "π/2": np.pi/2, "2π/3": 2*np.pi/3, "5π/6": 5*np.pi/6}
all_counts, labels = [], []

for name, theta in angles.items():
    q = QuantumCircuit(1, 1)
    q.ry(theta, 0)
    q.measure(0, 0)
    cts = run(q)
    all_counts.append(cts)
    labels.append(f"RY({name})  [expected P(1)={np.sin(theta/2)**2:.2f}]")
    print(f"RY({name}): {cts}")

plot_histogram(all_counts, legend=labels,
               color=["#001F66", "#003399", "#6A7FB5", "#E0884F", "#CC3300"],
               title="Different degrees of bias", figsize=(11, 6))

## Explanation
A qubit starts in |0⟩. The RY(θ) gate rotates the qubit's state around the Y axis of the Bloch sphere by angle θ, giving the state cos(θ/2)|0⟩ + sin(θ/2)|1⟩. Measurement probabilities are the squares of these amplitudes: P(0) = cos²(θ/2) and P(1) = sin²(θ/2). A small θ leaves the qubit close to |0⟩ (0 dominates), θ = π/2 gives a fair coin (the same as a Hadamard), and θ near π pushes it close to |1⟩ (1 dominates). With only 100 shots, the observed counts differ slightly from the predicted values due to statistical noise.

**Constraints met:** 1 qubit, measured, 1 gate before measurement (≤ 4), 100 shots.

## Challenge Question
**How can quantum gates change the probability of obtaining 0 or 1 when a qubit is measured?**

A qubit's state is a superposition α|0⟩ + β|1⟩, and the probability of measuring 0 or 1 is |α|² or |β|². Quantum gates are rotations of this state on the Bloch sphere, and they change the amplitudes α and β. A gate like X swaps them (flipping 0 and 1), H creates equal amplitudes (50/50), and a rotation gate such as RY(θ) sets them to cos(θ/2) and sin(θ/2) so that P(0) = cos²(θ/2) and P(1) = sin²(θ/2). By choosing the gate and the rotation angle, we control how much amplitude sits on |0⟩ versus |1⟩, and therefore how biased the measurement outcomes are.